In [7]:
from data_loader import load_mimic_tables                                      
                                                                                 
tables = load_mimic_tables()                                                                                                        
                                                                                 
# See all available tables:                                                    
print(tables.keys())  

/Users/huseyin/Codes/gpt-medic/explore/data_loader.py:26: DtypeWarning: Columns (4,6,7,8,9,10,11,12,13,15,16,17,18,21,23,24,25,26,27,28,29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  tables[table_name] = pd.read_csv(file, compression="gzip")


dict_keys(['hosp.poe', 'hosp.d_hcpcs', 'hosp.poe_detail', 'hosp.patients', 'hosp.diagnoses_icd', 'hosp.emar_detail', 'hosp.provider', 'hosp.prescriptions', 'hosp.drgcodes', 'hosp.d_icd_diagnoses', 'hosp.d_labitems', 'hosp.transfers', 'hosp.admissions', 'hosp.labevents', 'hosp.pharmacy', 'hosp.procedures_icd', 'hosp.hcpcsevents', 'hosp.services', 'hosp.d_icd_procedures', 'hosp.omr', 'hosp.emar', 'hosp.microbiologyevents', 'icu.datetimeevents', 'icu.caregiver', 'icu.ingredientevents', 'icu.inputevents', 'icu.procedureevents', 'icu.d_items', 'icu.chartevents', 'icu.icustays', 'icu.outputevents'])


In [8]:
import pandas as pd
from tokenizer.medication import MedicationTokenizer

gsn_atc = pd.read_csv('../data/gsn_atc_ndc_mapping.csv', dtype=str)

prescriptions = tables['hosp.prescriptions'].copy()
prescriptions['gsn'] = prescriptions['gsn'].astype(str).str.zfill(6)

# Drop rows with no GSN
prescriptions = prescriptions[prescriptions['gsn'] != '000nan']

# Map GSN → ATC
prescriptions = prescriptions.merge(
    gsn_atc[['gsn', 'atc']].drop_duplicates('gsn'),
    on='gsn',
    how='left'
).rename(columns={'atc': 'atc_code'})

print(f"Rows after drop:  {len(prescriptions)}")
print(f"ATC mapped:       {prescriptions['atc_code'].notna().sum()}")

med_tokenizer = MedicationTokenizer()
vocab = med_tokenizer.build_vocabulary(prescriptions.dropna(subset=['atc_code']))

print(f"Vocabulary size:  {len(vocab)}")
vocab

Rows after drop:  15568
ATC mapped:       15563
Vocabulary size:  190


{'ATC_4_A': 0,
 'ATC_4_B': 1,
 'ATC_4_C': 2,
 'ATC_4_D': 3,
 'ATC_4_E': 4,
 'ATC_4_F': 5,
 'ATC_4_G': 6,
 'ATC_4_H': 7,
 'ATC_4_M': 8,
 'ATC_4_X': 9,
 'ATC_A01': 10,
 'ATC_A02': 11,
 'ATC_A03': 12,
 'ATC_A04': 13,
 'ATC_A05': 14,
 'ATC_A06': 15,
 'ATC_A07': 16,
 'ATC_A09': 17,
 'ATC_A10': 18,
 'ATC_A11': 19,
 'ATC_A12': 20,
 'ATC_A16': 21,
 'ATC_B01': 22,
 'ATC_B02': 23,
 'ATC_B03': 24,
 'ATC_B05': 25,
 'ATC_C01': 26,
 'ATC_C02': 27,
 'ATC_C03': 28,
 'ATC_C04': 29,
 'ATC_C05': 30,
 'ATC_C07': 31,
 'ATC_C08': 32,
 'ATC_C09': 33,
 'ATC_C10': 34,
 'ATC_D01': 35,
 'ATC_D02': 36,
 'ATC_D03': 37,
 'ATC_D04': 38,
 'ATC_D06': 39,
 'ATC_D07': 40,
 'ATC_D10': 41,
 'ATC_D11': 42,
 'ATC_G02': 43,
 'ATC_G03': 44,
 'ATC_G04': 45,
 'ATC_H01': 46,
 'ATC_H02': 47,
 'ATC_H03': 48,
 'ATC_H04': 49,
 'ATC_H05': 50,
 'ATC_J01': 51,
 'ATC_J02': 52,
 'ATC_J05': 53,
 'ATC_J07': 54,
 'ATC_L01': 55,
 'ATC_L02': 56,
 'ATC_L03': 57,
 'ATC_M01': 58,
 'ATC_M02': 59,
 'ATC_M03': 60,
 'ATC_M04': 61,
 'ATC_M05': 62,
 '

In [13]:
tables['hosp.d_labitems'].head(10)

,itemid,label,fluid,category
0,50808,Free Calcium,Blood,Blood Gas
1,50826,Tidal Volume,Blood,Blood Gas
2,50813,Lactate,Blood,Blood Gas
3,52029,% Ionized Calcium,Blood,Blood Gas
4,50801,Alveolar-arterial Gradient,Blood,Blood Gas
5,50810,"Hematocrit, Calculated",Blood,Blood Gas
6,52025,Delete,Blood,Blood Gas
7,52031,Osmolality,Blood,Blood Gas
8,52022,"Albumin, Blood",Blood,Blood Gas
9,50806,"Chloride, Whole Blood",Blood,Blood Gas


In [ ]:
from tokenizer.lab import LabTokenizer

lab_tokenizer = LabTokenizer()
vocab = lab_tokenizer.build_vocabulary(tables['hosp.d_labitems'])

print(f"Vocabulary size: {len(vocab)}")

# Tokenize labevents
labevents_tokenized = lab_tokenizer.tokenize(tables['hosp.labevents'])
print()
labevents_tokenized[['itemid', 'tokenized_version']].head(10)

AttributeError: 'float' object has no attribute 'strip'